# Self-Designing Database: Monitoring System Demo

This notebook demonstrates the workload monitoring and feature extraction system.

In [3]:
# Import required modules
import sys
import os
sys.path.append('../src')

from monitoring.workload_monitor import WorkloadMonitor
from monitoring.db_inspector import DatabaseInspector
from feature_extractor.pipeline import FeatureExtractor

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

ModuleNotFoundError: No module named 'psycopg2'

## 1. Initialize Database Inspector

In [ ]:
# Database connection parameters
db_config = {
    'host': 'localhost',
    'port': 5432,
    'database': 'self_designing_db',
    'user': 'admin',
    'password': 'admin123'
}

# Create inspector
inspector = DatabaseInspector(db_config)
inspector.connect()

print("Database Inspector initialized")

NameError: name 'DatabaseInspector' is not defined

## 2. Create Database Snapshot

In [ ]:
# Create a snapshot
snapshot = inspector.create_snapshot("initial_snapshot")

# Display summary
summary = snapshot['summary']
print("Database Snapshot Summary:")
print(f"Total Tables: {summary['total_tables']}")
print(f"Total Indexes: {summary['total_indexes']}")
print(f"Used Indexes: {summary['used_indexes']}")
print(f"Unused Indexes: {summary['unused_indexes']}")
print(f"Total Database Size: {summary['total_database_size'] / (1024**3):.2f} GB")

## 3. Analyze Index Usage

In [ ]:
# Get all indexes
indexes = inspector.get_all_indexes()

# Create DataFrame for analysis
index_data = []
for idx in indexes:
    index_data.append({
        'name': idx.index_name,
        'table': idx.table_name,
        'type': idx.index_type,
        'size_mb': idx.index_size_bytes / (1024**2),
        'scans': idx.index_scan_count,
        'fetches': idx.index_tup_fetch,
        'usage_ratio': idx.usage_ratio,
        'efficiency': idx.efficiency
    })

df_indexes = pd.DataFrame(index_data)

# Display top 10 indexes by size
print("Top 10 Indexes by Size:")
print(df_indexes.nlargest(10, 'size_mb')[['name', 'table', 'size_mb', 'scans', 'usage_ratio']])

In [ ]:
# Visualize index usage
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Index size distribution
axes[0, 0].hist(df_indexes['size_mb'], bins=20, edgecolor='black')
axes[0, 0].set_xlabel('Index Size (MB)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Index Size Distribution')

# 2. Index usage vs size
axes[0, 1].scatter(df_indexes['size_mb'], df_indexes['scans'], alpha=0.6)
axes[0, 1].set_xlabel('Index Size (MB)')
axes[0, 1].set_ylabel('Number of Scans')
axes[0, 1].set_title('Index Usage vs Size')

# 3. Index type distribution
type_counts = df_indexes['type'].value_counts()
axes[1, 0].pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%')
axes[1, 0].set_title('Index Type Distribution')

# 4. Usage ratio distribution
axes[1, 1].hist(df_indexes['usage_ratio'].fillna(0), bins=20, edgecolor='black')
axes[1, 1].set_xlabel('Usage Ratio')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Index Usage Ratio Distribution')

plt.tight_layout()
plt.show()

## 4. Initialize Workload Monitor

In [ ]:
# Initialize workload monitor
monitor = WorkloadMonitor(db_config, poll_interval=10)

# Simulate monitoring for a short period
import asyncio

async def demo_monitoring():
    await monitor.connect()
    
    # Collect some metrics
    for _ in range(3):
        query_stats = await monitor.fetch_query_statistics()
        monitor.update_query_metrics(query_stats)
        await asyncio.sleep(5)
    
    return monitor.get_workload_summary()

# Run monitoring demo
summary = await demo_monitoring()

print("Workload Summary:")
print(f"Total Queries: {summary['total_queries']}")
print(f"Query Type Distribution: {summary['query_type_distribution']}")
print(f"Average Query Time: {summary['average_query_time']:.2f} ms")
print(f"P95 Latency: {summary['performance_metrics']['latency_percentiles']['p95']:.2f} ms")

## 5. Feature Extraction Pipeline

In [ ]:
# Initialize feature extractor
extractor = FeatureExtractor(normalize=True, feature_dim=256)

# Create example data
workload_example = {
    'query_type_distribution': {'select': 0.7, 'insert': 0.2, 'update': 0.1},
    'performance_metrics': {
        'latency_percentiles': {'p50': 10, 'p95': 50, 'p99': 100},
        'throughput_avg': 1000,
        'cache_hit_ratio_avg': 0.95
    },
    'temporal_patterns': {i: {'query_count': i*10, 'total_time': i*100} for i in range(24)},
    'top_tables': [{'table': f'table_{i}', 'accesses': i*100} for i in range(50)],
    'connections_ratio': 0.5,
    'lock_wait_ratio': 0.01
}

config_example = snapshot

performance_example = {
    'mean_latency': 25.5,
    'std_latency': 15.2,
    'min_latency': 1.0,
    'max_latency': 200.0,
    'transaction_latency': 30.0,
    'read_throughput': 800,
    'write_throughput': 200,
    'total_throughput': 1000,
    'cpu_utilization': 0.65,
    'memory_utilization': 0.75,
    'disk_io_utilization': 0.45,
    'error_rate': 0.001,
    'timeout_rate': 0.0005,
    'replication_lag': 0.0
}

# Extract features
state_vector = extractor.extract_state_vector(
    workload_example,
    config_example,
    performance_example
)

print(f"State vector shape: {state_vector.shape}")
print(f"State vector dimension: {len(state_vector)}")
print(f"\nFirst 20 features:")
print(state_vector[:20])

In [ ]:
# Visualize feature distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Feature value distribution
axes[0].hist(state_vector, bins=30, edgecolor='black')
axes[0].set_xlabel('Feature Value')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Feature Value Distribution')

# 2. Cumulative distribution
sorted_features = np.sort(state_vector)
cumulative = np.arange(1, len(sorted_features) + 1) / len(sorted_features)
axes[1].plot(sorted_features, cumulative, linewidth=2)
axes[1].set_xlabel('Feature Value')
axes[1].set_ylabel('Cumulative Probability')
axes[1].set_title('Cumulative Distribution')
axes[1].grid(True, alpha=0.3)

# 3. Feature importance (placeholder - would use actual model)
feature_importance = np.abs(state_vector)
top_indices = np.argsort(feature_importance)[-20:]
feature_names = extractor.get_feature_names()

axes[2].barh(range(20), feature_importance[top_indices])
axes[2].set_yticks(range(20))
axes[2].set_yticklabels([feature_names[i] for i in top_indices], fontsize=8)
axes[2].set_xlabel('Importance (abs value)')
axes[2].set_title('Top 20 Features by Importance')

plt.tight_layout()
plt.show()

## 6. Export Metrics for Prometheus

In [ ]:
# Export metrics in Prometheus format
prometheus_metrics = monitor.export_metrics(format='prometheus')

print("Prometheus Metrics (first 20 lines):\n")
lines = prometheus_metrics.split('\n')[:20]
for line in lines:
    print(line)

## 7. Clean Up

In [ ]:
# Disconnect from database
inspector.disconnect()

print("Clean up complete. All connections closed.")

## Summary

This demo showed:
1. Database inspection and snapshot creation
2. Index usage analysis and visualization
3. Workload monitoring setup
4. Feature extraction pipeline
5. Metrics export for monitoring

The system is now ready for integration with the RL environment.